# Hotdog / Not Hotdog: the CNN

Fine-tunes an ImageNet-pretrained **ConvNeXt-Tiny** to tell hot dogs from everything else and exports it for the
CNN server. Run it here cell by cell, or headless with `./run.sh train` (~20 min on an Apple Silicon Mac).

**Data:** [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/), a free ~5 GB download with no
account needed. Training uses its 750 training hot dogs plus ~2 200 other dishes, two thirds of them look-alikes (fries,
prime rib, tacos …). The test set is 250 hot dogs + 250 other dishes from Food-101's separate test split.

`SMOKE=1` runs 2 epochs on synthetic images without any download, to check the pipeline.

## 1. Setup

In [ ]:
import os, sys, json, math, random, time, platform, warnings
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

sys.path.insert(0, str(Path.cwd()))  # run from cnn/: hotdog_model.py and hotdog_data.py live next to this notebook
from hotdog_model import (CLASSES, IMG_SIZE, RESIZE_SIZE, IMAGENET_MEAN, IMAGENET_STD, ARCHS,
                          build_model, describe, param_groups, set_backbone_trainable, cam_layer,
                          count_params, pick_device, sync_device)
from hotdog_data import (HotdogDataset, open_image, load_kaggle, load_hf, load_folder, load_food101, load_food101_splits,
                         drop_overlaps, make_synthetic)

SEED = 42
def seed_everything(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()

device = pick_device(os.environ.get("DEVICE") or None)
print(f"python {platform.python_version()} | torch {torch.__version__} | device: {device}")

## 2. Configuration

Each setting can also be set as an environment variable of the same name.

| Setting | Options |
|---|---|
| `DATA_SOURCE` | `food101` (default, no account) · `kaggle` (SeeFood, needs Kaggle credentials) · `hf` (a Hugging Face dataset) · `folder` (`data/{train,test}/{hot_dog,not_hot_dog}`) |
| `ADD_FOOD101` | for `kaggle` / `hf` / `folder` only: `1` (default) adds Food-101 to training · `0`: source only |
| `ARCH` | `convnext_tiny` (default) · `efficientnet_b0` · `mobilenet_v3_large` (faster, lighter) |
| `DEVICE` | auto (CUDA → Apple MPS → CPU) or e.g. `cpu` |

In [ ]:
env = os.environ.get
SMOKE        = env("SMOKE", "0") == "1"              # tiny synthetic run, no downloads

DATA_SOURCE  = env("DATA_SOURCE", "food101")         # "food101" | "kaggle" | "hf" | "folder"
KAGGLE_ID    = "dansbecker/hot-dog-not-hot-dog"
HF_DATASET, HF_IMAGE_COL, HF_LABEL_COL, HF_TEST_SPLIT = "truepositive/hotdog_nothotdog", "image", "label", "test"
DATA_DIR     = Path(env("DATA_DIR", "data"))         # local folder source + Food-101 download location

ADD_FOOD101  = env("ADD_FOOD101", "1") == "1" and DATA_SOURCE != "food101"  # extra training data for other sources
FOOD101_HOTDOGS        = 1000                         # all of Food-101's hot_dog class
FOOD101_HARD_PER_CLASS = 60                           # x ~25 look-alike classes  (~1 500 negatives)
FOOD101_OTHER_PER_CLASS = 10                          # x ~75 other classes       (~  750 negatives)

ARCH         = env("ARCH", "convnext_tiny")
assert ARCH in ARCHS, f"ARCH must be one of {ARCHS}"
PRETRAINED   = env("PRETRAINED", "1") == "1" and not SMOKE   # ImageNet weights (smoke mode never downloads)

EPOCHS, FREEZE_EPOCHS, PATIENCE = 12, 2, 4            # phase 1: head only; phase 2: everything
LR, FINETUNE_LR, BACKBONE_LR_MULT = 1e-3, 5e-4, 0.2   # backbone fine-tunes at 1e-4
BATCH_SIZE   = 32
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.05                                # light; calibration is done properly below
WARMUP_EPOCHS = 1
VAL_FRACTION = 0.15
USE_AMP      = True                                   # fp16 autocast on CUDA / MPS when supported
NUM_WORKERS  = 0 if SMOKE else min(8, os.cpu_count() or 1)

ARTIFACTS_DIR = Path(env("CNN_ARTIFACTS", "artifacts"))

if SMOKE:
    EPOCHS, FREEZE_EPOCHS, BATCH_SIZE, WARMUP_EPOCHS, PATIENCE = 2, 1, 8, 0, 2
    ADD_FOOD101 = False
print(f"SMOKE={SMOKE} | data={'synthetic' if SMOKE else DATA_SOURCE}{' + Food-101' if ADD_FOOD101 else ''} | "
      f"ARCH={ARCH} ({describe(ARCH)}) | pretrained={PRETRAINED} | epochs={EPOCHS} | workers={NUM_WORKERS}")

## 3. Data

The test split is only used in section 8. Food-101's train and test splits never overlap. With another source plus
`ADD_FOOD101`, Food-101 images matching any of that source's images (same file id or near-identical perceptual hash)
are dropped first. 15 % of the training pool becomes the validation set, used for model selection and calibration.

In [ ]:
print("==> Data")
if SMOKE:
    (train_x, train_y), (test_x, test_y) = make_synthetic(48, 1), make_synthetic(16, 2)
    data_desc = "synthetic (SMOKE)"
elif DATA_SOURCE == "food101":
    (train_x, train_y, train_cls), (test_x, test_y) = load_food101_splits(
        DATA_DIR, FOOD101_HARD_PER_CLASS, FOOD101_OTHER_PER_CLASS, seed=SEED)
    data_desc = "food101"
elif DATA_SOURCE == "kaggle":
    (train_x, train_y), (test_x, test_y) = load_kaggle(KAGGLE_ID); data_desc = f"kaggle:{KAGGLE_ID}"
elif DATA_SOURCE == "hf":
    (train_x, train_y), (test_x, test_y) = load_hf(HF_DATASET, HF_IMAGE_COL, HF_LABEL_COL, HF_TEST_SPLIT); data_desc = f"hf:{HF_DATASET}"
elif DATA_SOURCE == "folder":
    (train_x, train_y), (test_x, test_y) = load_folder(DATA_DIR); data_desc = f"folder:{DATA_DIR}"
else:
    raise ValueError(DATA_SOURCE)

if not test_x:   # source without a test split: carve one out before anything else touches the data
    train_x, test_x, train_y, test_y = train_test_split(train_x, train_y, test_size=0.2, stratify=train_y, random_state=SEED)
if DATA_SOURCE == "food101" and not SMOKE:
    train_src = [f"food101:{c}" for c in train_cls]
else:
    train_src = [DATA_SOURCE if not SMOKE else "synthetic"] * len(train_x)

if ADD_FOOD101:
    f_x, f_y, f_cls = load_food101(DATA_DIR, FOOD101_HOTDOGS, FOOD101_HARD_PER_CLASS, FOOD101_OTHER_PER_CLASS, seed=SEED)
    t0 = time.time()
    f_x, f_y, f_cls, dropped = drop_overlaps(f_x, f_y, f_cls, protected=list(test_x) + list(train_x))
    print(f"Food-101: kept {len(f_x)} images, removed overlaps with the Kaggle data: {dropped or 'none'} ({time.time()-t0:.0f}s)")
    train_x, train_y, train_src = list(train_x) + f_x, list(train_y) + f_y, train_src + [f"food101:{c}" for c in f_cls]
    data_desc += " + food101"

idx_tr, idx_val = train_test_split(np.arange(len(train_x)), test_size=VAL_FRACTION,
                                   stratify=[f"{y}|{s.split(':')[0]}" for y, s in zip(train_y, train_src)], random_state=SEED)
val_x, val_y, val_src = [train_x[i] for i in idx_val], [train_y[i] for i in idx_val], [train_src[i] for i in idx_val]
train_x, train_y, train_src = [train_x[i] for i in idx_tr], [train_y[i] for i in idx_tr], [train_src[i] for i in idx_tr]

for name, ys in [("train", train_y), ("val", val_y), ("test", test_y)]:
    c = np.bincount(ys, minlength=2)
    print(f"{name:5s}: {len(ys):5d} images  hotdog={c[0]:5d}  not_hotdog={c[1]:5d}")

### A look at the data

In [ ]:
def show_grid(items, titles, ncols=6, size=2.2, suptitle=None):
    n = len(items); nrows = max(1, math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size + (0.4 if suptitle else 0)))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, im, t in zip(np.atleast_1d(axes).ravel(), items, titles):
        ax.imshow(im); ax.set_title(t, fontsize=8)
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

idx = random.Random(0).sample(range(len(train_x)), k=min(12, len(train_x)))
show_grid([open_image(train_x[i]).resize((160, 160)) for i in idx],
          [f"{CLASSES[train_y[i]]}\n{train_src[i]}" for i in idx], suptitle="Training samples (label / source)")

## 4. Augmentation and loaders

Training images get random crops, flips, TrivialAugment and random erasing; evaluation uses `Resize(256) +
CenterCrop(224)`, like the server. A balanced sampler draws every batch 50/50 hot dog / not hot dog.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.35, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])
eval_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
view_tf = transforms.Compose([transforms.Resize(RESIZE_SIZE), transforms.CenterCrop(IMG_SIZE)])  # for plots

counts = np.bincount(train_y, minlength=2)
sampler = WeightedRandomSampler(weights=[1.0 / counts[y] for y in train_y], num_samples=len(train_y), replacement=True)

pin = device.type == "cuda"
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
train_dl = DataLoader(HotdogDataset(train_x, train_y, train_tf), batch_size=BATCH_SIZE, sampler=sampler,
                      drop_last=len(train_x) > BATCH_SIZE, **loader_kw)
val_dl   = DataLoader(HotdogDataset(val_x, val_y, eval_tf), batch_size=BATCH_SIZE * 2, **loader_kw)
test_ds  = HotdogDataset(test_x, test_y, eval_tf)
test_dl  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, **loader_kw)

def denorm(t):
    return (t * torch.tensor(IMAGENET_STD)[:, None, None] + torch.tensor(IMAGENET_MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0).numpy()

xb, yb = next(iter(train_dl))
print("one batch, hotdog share:", float((yb == 0).float().mean()))
show_grid([denorm(t) for t in xb[:12]], [CLASSES[y] for y in yb[:12]], suptitle="Augmented training batch")

## 5. Model

An ImageNet-pretrained CNN with a new 2-way head.

In [ ]:
model = build_model(ARCH, num_classes=len(CLASSES), pretrained=PRETRAINED).to(device)
total, _ = count_params(model)
print(f"{ARCH}: {describe(ARCH)} | {total:,} parameters | pretrained weights loaded: {PRETRAINED}")

## 6. Training

Two phases: the new head alone, then the whole network with the pretrained part at a lower learning rate. AdamW with
a cosine schedule; fp16 where supported. The epoch with the best validation accuracy (ties: lower loss) is kept, and
training stops early after `PATIENCE` epochs without improvement.

In [ ]:
print("==> Training")
def make_autocast():
    if not USE_AMP or device.type not in ("cuda", "mps"):
        return nullcontext, None
    try:
        model.eval()  # probe in eval mode so BatchNorm running stats are untouched
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16):
            model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
    except Exception as e:
        print(f"autocast unavailable on {device.type} ({e}); using fp32")
        return nullcontext, None
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    return (lambda: torch.autocast(device_type=device.type, dtype=torch.float16)), scaler

autocast, scaler = make_autocast()
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("mixed precision:", autocast is not nullcontext)

def make_optim(lr, epochs, warmup, backbone_mult):
    opt = torch.optim.AdamW(param_groups(model, lr, backbone_mult), weight_decay=WEIGHT_DECAY)
    total = max(1, epochs * len(train_dl)); warm = warmup * len(train_dl)
    def lr_lambda(step):
        if step < warm: return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

@torch.no_grad()
def predict_logits(dl, m=None):
    m = m or model; m.eval(); out, ys = [], []
    for x, y in dl:
        with autocast():
            out.append(m(x.to(device)).float().cpu())
        ys.append(y)
    return torch.cat(out), torch.cat(ys)

def evaluate(dl):
    logits, ys = predict_logits(dl)
    loss = F.cross_entropy(logits, ys).item()          # plain NLL, comparable across runs
    return loss, accuracy_score(ys.numpy(), logits.argmax(1).numpy())

history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc", "lr")}
best = {"key": (-1.0, -math.inf), "val_acc": -1.0, "epoch": -1, "state": None}

def run_phase(name, epochs, lr, warmup, backbone_mult):
    opt, sched = make_optim(lr, epochs, warmup, backbone_mult)
    stale = 0
    for ep in range(epochs):
        model.train(); loss_sum, correct, n = 0.0, 0, 0
        for x, y in tqdm(train_dl, desc=f"[{name}] epoch {ep + 1}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast():
                logits = model(x)
                loss = criterion(logits.float(), y)
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 2.0); scaler.step(opt); scaler.update()
            else:
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
            sched.step()
            loss_sum += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); n += len(y)
        vl, va = evaluate(val_dl)
        for k, v in [("train_loss", loss_sum / n), ("train_acc", correct / n), ("val_loss", vl), ("val_acc", va), ("lr", sched.get_last_lr()[0])]:
            history[k].append(v)
        key = (round(va, 4), -vl)
        improved = key > best["key"]
        if improved:
            best.update(key=key, val_acc=va, epoch=len(history["val_acc"]),
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            stale = 0
        else:
            stale += 1
        print(f"[{name}] epoch {ep + 1:3d}  train loss {loss_sum / n:.4f} acc {correct / n:.3f} | "
              f"val loss {vl:.4f} acc {va:.3f}{'  *' if improved else ''}")
        if stale >= PATIENCE:
            print(f"early stopping: no val improvement for {PATIENCE} epochs"); break

t_start = time.time()
set_backbone_trainable(model, False)
print(f"phase 1: head only ({count_params(model)[1]:,} trainable params)")
run_phase("head", FREEZE_EPOCHS, LR, warmup=0, backbone_mult=0.0)
set_backbone_trainable(model, True)
print(f"phase 2: full fine-tune ({count_params(model)[1]:,} trainable params, backbone LR {FINETUNE_LR * BACKBONE_LR_MULT:.0e})")
run_phase("finetune", EPOCHS - FREEZE_EPOCHS, FINETUNE_LR, warmup=WARMUP_EPOCHS, backbone_mult=BACKBONE_LR_MULT)
train_minutes = (time.time() - t_start) / 60
model.load_state_dict(best["state"])
print(f"\nbest val acc {best['val_acc']:.3f} at epoch {best['epoch']} | training took {train_minutes:.1f} min")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history["train_loss"]) + 1)
a1.plot(ep, history["train_loss"], label="train"); a1.plot(ep, history["val_loss"], label="val")
a1.set(title="Loss (train includes label smoothing)", xlabel="epoch"); a1.legend()
a2.plot(ep, history["train_acc"], label="train"); a2.plot(ep, history["val_acc"], label="val")
a2.axvline(best["epoch"], ls="--", c="gray", lw=1, label="best")
if not SMOKE: a2.axvspan(0.5, FREEZE_EPOCHS + 0.5, color="gray", alpha=0.1, label="head only")
a2.set(title="Accuracy", xlabel="epoch", ylim=(0, 1.02)); a2.legend()
plt.tight_layout(); plt.show()

## 7. Calibration

Temperature scaling fits one number `T` on the validation set so that `softmax(logits / T)` is honest: "90 % sure"
means right ~90 % of the time. Accuracy is unchanged. ECE is the average gap between confidence and accuracy. The
server applies `T`.

In [ ]:
def ece(probs, ys, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == ys).astype(float); e = 0.0
    for lo in np.linspace(0, 1, n_bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / n_bins)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

val_logits, val_ys = predict_logits(val_dl)
log_t = torch.zeros(1, requires_grad=True)
opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
def closure():
    opt_t.zero_grad(); l = F.cross_entropy(val_logits / log_t.exp(), val_ys); l.backward(); return l
opt_t.step(closure)
TEMPERATURE = float(log_t.exp().clamp(0.05, 20).item())

vy = val_ys.numpy()
ece_before = ece(val_logits.softmax(1).numpy(), vy)
ece_after  = ece((val_logits / TEMPERATURE).softmax(1).numpy(), vy)
print(f"temperature T = {TEMPERATURE:.3f}  |  val ECE {ece_before:.3f} -> {ece_after:.3f}")

## 8. Evaluation

The held-out test images, scored like the server does. Precision, recall and F1 treat **hotdog** as positive.

In [ ]:
print("==> Evaluation")
test_logits, test_ys = predict_logits(test_dl)
y_prob = (test_logits / TEMPERATURE).softmax(1).numpy()
y_true, y_pred = test_ys.numpy(), y_prob.argmax(1)
test_acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, pos_label=0, average="binary", zero_division=0)
try:
    auc = roc_auc_score(y_true == 0, y_prob[:, 0])
except ValueError:
    auc = float("nan")
metrics = {"val_acc": round(float(best["val_acc"]), 4), "test_acc": round(float(test_acc), 4),
           "test_precision": round(float(prec), 4), "test_recall": round(float(rec), 4), "test_f1": round(float(f1), 4),
           "test_roc_auc": round(float(auc), 4), "test_ece": round(ece(y_prob, y_true), 4)}
print(json.dumps(metrics, indent=2))
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASSES, zero_division=0))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]), display_labels=CLASSES).plot(ax=a1, cmap="Blues", colorbar=False)
a1.set_title(f"Test confusion matrix (acc {test_acc:.3f})")
# reliability diagram
conf, correct = y_prob.max(1), (y_pred == y_true)
bins = np.linspace(0.5, 1.0, 6); mids, accs, ns = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf <= hi)
    if m.any(): mids.append(conf[m].mean()); accs.append(correct[m].mean()); ns.append(m.sum())
a2.plot([0.5, 1], [0.5, 1], "--", c="gray", lw=1, label="perfectly calibrated")
a2.plot(mids, accs, "o-", label="model"); [a2.annotate(str(n), (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8) for x, y, n in zip(mids, accs, ns)]
a2.set(title="Reliability (test, calibrated)", xlabel="confidence", ylabel="accuracy", xlim=(0.5, 1), ylim=(0, 1.08)); a2.legend(loc="lower right")
plt.tight_layout(); plt.show()

### Mistakes and Grad-CAM

Grad-CAM highlights the regions that drove a prediction. It should light up the sausage and bun, not the plate.

In [ ]:
def gradcam(x, target_cls):
    # Grad-CAM for a single (1, 3, H, W) tensor, returns an HxW map in [0, 1]
    layer, store = cam_layer(model), {}
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__("act", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("grad", go[0]))
    try:
        model.eval(); model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = model(x.to(device).requires_grad_(True))
            out[0, target_cls].backward()
        w = store["grad"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * store["act"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().float().cpu().numpy()
    finally:
        h1.remove(); h2.remove(); model.zero_grad(set_to_none=True)

def overlay(i, cls):
    base = np.asarray(view_tf(open_image(test_x[i]))) / 255.0
    cam = gradcam(test_ds[i][0].unsqueeze(0), cls)
    return np.clip(0.55 * base + 0.45 * plt.cm.jet(cam)[..., :3], 0, 1)

wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} of {len(y_true)} test images misclassified")
if len(wrong):
    wrong = wrong[np.argsort(-y_prob[wrong].max(1))][:12]      # most confident mistakes first
    show_grid([open_image(test_x[i]).resize((160, 160)) for i in wrong],
              [f"true {CLASSES[y_true[i]]}\npred {CLASSES[y_pred[i]]} {y_prob[i].max():.2f}" for i in wrong],
              suptitle="Misclassified test images (most confident first)")
    show_grid([overlay(i, y_pred[i]) for i in wrong], [f"why '{CLASSES[y_pred[i]]}'?" for i in wrong],
              suptitle="Grad-CAM: what drove each wrong prediction")

hits = np.where((y_true == 0) & (y_pred == 0))[0]
if len(hits):
    hits = hits[np.argsort(-y_prob[hits, 0])][:6]
    show_grid([overlay(i, 0) for i in hits], [f"hotdog {y_prob[i, 0]:.2f}" for i in hits],
              suptitle="Grad-CAM on confident correct hotdogs (should light up the sausage + bun)")

### Latency

Batch size 1, forward pass only: what the server reports as `latency_ms`.

In [ ]:
@torch.no_grad()
def benchmark(m, n_warmup=10, n_runs=100):
    m.eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(n_warmup): m(x)
    sync_device(device); times = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); m(x); sync_device(device); times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))

lat_p50, lat_p95 = benchmark(model, *((2, 5) if SMOKE else (10, 100)))
print(f"{ARCH} on {device}: p50 {lat_p50:.2f} ms/image, p95 {lat_p95:.2f} ms/image (batch 1, fp32)")

## 9. Export

Writes `hotdog_cnn.pt` (weights), `hotdog_cnn.ts.pt` (TorchScript fallback) and `model_meta.json` (architecture,
preprocessing, temperature, metrics) to `artifacts/`, where the server loads them on its next start (`./run.sh`).

In [ ]:
print("==> Export")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

cpu_model = build_model(ARCH, num_classes=len(CLASSES)).eval()
cpu_model.load_state_dict(best["state"])
torch.save(cpu_model.state_dict(), ARTIFACTS_DIR / "hotdog_cnn.pt")

try:
    with warnings.catch_warnings():  # torch.jit is deprecated in recent torch but still works
        warnings.simplefilter("ignore")
        ts = torch.jit.trace(cpu_model, torch.zeros(1, 3, IMG_SIZE, IMG_SIZE))
        ts.save(str(ARTIFACTS_DIR / "hotdog_cnn.ts.pt"))
    ts_ok = True
except Exception as e:  # TorchScript is optional; the state_dict is the primary format
    print("TorchScript export failed:", e); ts_ok = False

src_counts = {}
for s in train_src + val_src:
    k = s.split(":")[0]; src_counts[k] = src_counts.get(k, 0) + 1
meta = {
    "arch": ARCH, "arch_description": describe(ARCH), "pretrained": PRETRAINED,
    "classes": CLASSES, "img_size": IMG_SIZE, "resize_size": RESIZE_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
    "temperature": round(TEMPERATURE, 5),
    "metrics": metrics, "best_epoch": best["epoch"], "epochs_run": len(history["val_acc"]), "train_minutes": round(train_minutes, 2),
    "params": count_params(cpu_model)[0], "latency_ms_p50": round(lat_p50, 3), "latency_device": device.type,
    "dataset": data_desc, "train_pool_by_source": src_counts,
    "n_train": len(train_y), "n_val": len(val_y), "n_test": len(test_y),
    "smoke": SMOKE, "torch": torch.__version__, "torchscript": ts_ok,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(ARTIFACTS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
for f in sorted(ARTIFACTS_DIR.glob("*.*")):
    print(f"{f.name:22s} {f.stat().st_size / 1e6:7.2f} MB")

In [ ]:
# Reload the way the server does and check the predictions match
reloaded = build_model(meta["arch"], num_classes=2)
reloaded.load_state_dict(torch.load(ARTIFACTS_DIR / "hotdog_cnn.pt", map_location="cpu", weights_only=True))
reloaded.eval()
x = torch.stack([test_ds[i][0] for i in range(min(8, len(test_ds)))])
with torch.no_grad():
    a = (reloaded(x) / meta["temperature"]).softmax(1)
    b = (cpu_model(x) / TEMPERATURE).softmax(1)
assert torch.allclose(a, b, atol=1e-5), "reloaded model disagrees"
if ts_ok:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        assert torch.allclose((torch.jit.load(str(ARTIFACTS_DIR / "hotdog_cnn.ts.pt"))(x) / TEMPERATURE).softmax(1), b, atol=1e-4)
print("reload check passed")